# Phase 0 — Verify the Kaggle inputs

Answers the four verification questions in `docs/05_dataset_card.md` before any
compute is spent. **Two of them are load-bearing** — if Q1 fails there is no version
of this thesis, and finding that out now costs minutes instead of weeks.

## Notebook settings (right-hand panel)

| Setting | Value |
|---|---|
| Accelerator | **None** — this notebook uses no GPU quota |
| Persistence | No persistence |
| Internet | **On** (needed for the git clone) |
| Environment | **Pin to original environment** |

## Inputs to add

All six, this once. After Phase 1 nothing reads the raw datasets again.

`eyepacs-original` · `ddr-dataset-credits-to-authors` · `idrid` ·
`aptos2019-blindness-detection` · `messidor2preprocess` · `messidor2-dr-grades`

## Exit condition

Q1 and Q2 answered **in writing** in `docs/05_dataset_card.md`.

## 1 · Pull the code

In [ ]:
import shutil, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always take a clean checkout

# Private repo? Store a GitHub PAT under Add-ons -> Secrets as GH_TOKEN.
# Public repo? Delete the try/except and just clone the plain URL.
url = "https://github.com/kazimab1/DR-New.git"
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
    url = url.replace("https://", f"https://{token}@")
    print("cloning with GH_TOKEN")
except Exception:
    print("no GH_TOKEN secret found - cloning anonymously (works if the repo is public)")

!git clone --depth 1 -b claude/charming-faraday-a02cx9 {url} {REPO_DIR} 2>&1 | tail -2

sys.path.insert(0, str(REPO_DIR / "scripts"))
assert (REPO_DIR / "scripts/build_cache.py").exists(), "clone failed - check the token or branch name"

# Print the commit actually in use. A stale checkout is the single most common
# cause of a confusing failure downstream: the notebook cell is new, the scripts
# on disk are not.
import subprocess
_sha = subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("repo ready at", REPO_DIR)
print("checked out:", _sha)

In [ ]:
from pathlib import Path
import os, re, json

INPUT = Path("/kaggle/input")

def _norm(s):
    return s.lower().replace("-", "").replace("_", "").replace("%20", "")

def dataset_roots(max_depth=3):
    """Candidate dataset directories, shallowest first.

    Kaggle does not always mount datasets as direct children of /kaggle/input --
    they can sit under competitions/ and datasets/ wrappers. Breadth-first so a
    shallower match always wins over a nested subfolder of the same name.
    """
    level, out = [INPUT], []
    for _ in range(max_depth):
        nxt = []
        for d in level:
            try:
                children = sorted(c for c in d.iterdir() if c.is_dir())
            except OSError:
                continue
            out.extend(children)
            nxt.extend(children)
        level = nxt
    return out

EXCLUDE_ROOTS = []      # set to [CACHE] once the cache is located

def _under(path, root):
    try:
        path.relative_to(root)
        return True
    except ValueError:
        return False

def find_mount(*keywords, required=True):
    """Locate a mounted dataset by keyword, so a renamed mirror doesn't break the notebook.

    Anything under EXCLUDE_ROOTS is skipped. The cache contains directories named
    ddr, eyepacs, idrid, aptos and messidor2 -- exactly the keywords searched for --
    so without this a raw-dataset lookup can land inside the cache instead.
    """
    for d in dataset_roots():
        if any(_under(d, r) for r in EXCLUDE_ROOTS):
            continue
        if all(_norm(k) in _norm(d.name) for k in keywords):
            return d
    if required:
        print(f"  !! NOT MOUNTED: {' + '.join(keywords)}")
        print("     Use 'Add Input' in the right-hand panel. Candidate dirs seen:")
        for d in dataset_roots(2)[:25]:
            print(f"       - {d.relative_to(INPUT)}")
    return None

def find_any(*keyword_sets, label="", required=True):
    """Try several keyword spellings. Mirrors title themselves inconsistently:
    IDRiD ships as 'idrid-dataset' or 'indian-diabetic-retinopathy-image-dataset'."""
    for kws in keyword_sets:
        hit = find_mount(*kws, required=False)
        if hit:
            return hit
    if required:
        print(f"  !! NOT MOUNTED: {label or keyword_sets[0]}")
        print("     Currently mounted:")
        for d in sorted(INPUT.iterdir()):
            print(f"       - {d.name}")
    return None

def match_channel(dirname):
    """Map a mask directory name to a lesion channel.

    DDR uses MA/HE/EX/SE; IDRiD uses '1. Microaneurysms', '2. Haemorrhages',
    '3. Hard Exudates', '4. Soft Exudates', '5. Optic Disc'. Match on meaning so
    one function covers both.
    """
    n = re.sub(r"^\d+\.\s*", "", dirname.lower().strip())
    n = n.replace("%20", " ")
    if n == "ma" or "microaneurysm" in n:
        return "microaneurysm"
    if n == "he" or "haemorrhage" in n or "hemorrhage" in n:
        return "haemorrhage"
    if n == "ex" or ("hard" in n and "exudate" in n):
        return "hard_exudate"
    if n == "se" or ("soft" in n and "exudate" in n) or "cotton" in n:
        return "soft_exudate"
    if n == "od" or "optic disc" in n:
        return "optic_disc"
    return None

MASK_EXT = {".tif", ".tiff", ".png", ".gif", ".bmp", ".jpg", ".jpeg"}
LESION4 = ["microaneurysm", "haemorrhage", "hard_exudate", "soft_exudate"]

def scan_mask_dirs(root, require=""):
    """Find mask directories under root, keyed by lesion channel."""
    found = {}
    for d in root.rglob("*"):
        if not d.is_dir():
            continue
        if require and require not in str(d).lower():
            continue
        channel = match_channel(d.name)
        if not channel:
            continue
        files = [f for f in d.rglob("*") if f.suffix.lower() in MASK_EXT]
        if files:
            found.setdefault(channel, []).append((str(d), len(files)))
    return found

def du(path, cap=40000):
    """Rough size + file count, capped so it stays fast on huge mounts."""
    total = n = 0
    for i, f in enumerate(Path(path).rglob("*")):
        if i > cap:
            return total, n, True
        if f.is_file():
            total += f.stat().st_size
            n += 1
    return total, n, False

print("helpers ready")

## 2 · What machine did we get?

In [ ]:
import multiprocessing, shutil, sys
print("python     ", sys.version.split()[0])
print("cpu cores  ", multiprocessing.cpu_count())
!free -g | head -2
!df -h /kaggle/working | tail -1
for mod in ("cv2", "numpy", "pandas", "torch"):
    try:
        m = __import__(mod)
        print(f"{mod:10s} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"{mod:10s} NOT INSTALLED")

## 3 · Which datasets are actually mounted?

In [ ]:
print("mounted under /kaggle/input:\n")
for d in sorted(INPUT.iterdir()):
    size, n, capped = du(d)
    print(f"  {d.name:55s} {size/2**30:7.2f} GiB  {n:>7}{'+' if capped else ''} files")

# Kaggle may nest real datasets under competitions/ and datasets/ wrappers, so
# list what discovery will actually match against.
print("\ncandidate dataset roots (what find_mount searches):\n")
for d in dataset_roots(2):
    depth = len(d.relative_to(INPUT).parts)
    if depth == 1 and any(c.is_dir() for c in d.iterdir()):
        continue                      # a wrapper; its children are listed below
    print(f"  [{depth}] {d.relative_to(INPUT)}")

---
## Q1 · Does the DDR mirror ship lesion segmentation masks?

**This is the blocking question.** The evidence pathway (M2) trains on DDR's
pixel-level lesion annotations. If this mirror carries only `DR_grading/`, M2 has no
training data — and M2 *is* the thesis.

Looking for a `lesion_segmentation/` tree with four channels (MA, HE, EX, SE). The
canonical DDR segmentation subset is **757 images**, split 383 train / 149 valid /
225 test.

DDR mirrors vary — some carry only `DR_grading/`. The cell below **discovers** the
layout rather than assuming it, so it works whichever mirror you mounted.

In [ ]:
ddr = find_mount("ddr")
if ddr:
    print("top-level layout:")
    for p in sorted(ddr.rglob("*")):
        rel = p.relative_to(ddr)
        if p.is_dir() and len(rel.parts) <= 3:
            print("   ", rel)

In [ ]:
ddr_masks, ddr_seg_images = {}, []
if ddr:
    seg_roots = [p for p in ddr.rglob("*") if p.is_dir()
                 and "segmentation" in p.name.lower()]
    print("segmentation roots found:", [str(p.relative_to(ddr)) for p in seg_roots] or "NONE")

    scanned = scan_mask_dirs(ddr, require="segmentation")
    for channel in LESION4:
        entries = scanned.get(channel, [])
        if entries:
            ddr_masks[channel] = sorted(d for d, _ in entries)
            print(f"  {channel:15s} {sum(n for _, n in entries):5d} masks "
                  f"in {len(entries)} dir(s)")
        else:
            print(f"  {channel:15s}     0 masks   *** MISSING ***")

    imgs = [p for p in ddr.rglob("*") if p.is_dir() and p.name.lower() == "image"
            and "segmentation" in str(p).lower()]
    ddr_seg_images = sorted(str(p) for p in imgs)
    print("\nsegmentation image dirs:", ddr_seg_images or "NONE")

    # DDR ships grading labels as train/valid/test .txt, not a CSV. Phase 2
    # needs to know which, so record it now.
    grading = next((d for d in ddr.rglob("*") if d.is_dir()
                    and "grading" in d.name.lower()), None)
    if grading:
        labels = [f for f in grading.rglob("*") if f.suffix.lower() in {".txt", ".csv"}]
        print(f"\nDR_grading label files: {[f.name for f in labels] or 'NONE'}")
        if labels:
            head = labels[0].read_text(errors="ignore").splitlines()[:3]
            print(f"  format of {labels[0].name}: {head}")
        splits = sorted(d.name for d in grading.iterdir() if d.is_dir())
        print(f"  DR_grading split dirs: {splits}")

Q1_OK = len(ddr_masks) == 4
print("\n" + ("=" * 62))
if Q1_OK:
    print("Q1 PASS - all four lesion channels present. Evidence pathway is viable.")
else:
    print("Q1 FAIL - lesion masks incomplete or absent.")
    print("STOP. Find another DDR mirror before spending any compute.")
    print("Without pixel masks, M2 cannot be trained and the thesis has no subject.")
print("=" * 62)

---
## Q2 · What preprocessing has `messidor2preprocess` already applied?

If these images are already contrast-normalised or Ben-Graham processed, part of the
measured EyePACS→Messidor-2 "domain gap" is a preprocessing artefact rather than a
real distribution shift. Either match the processing across all sources or report the
confound — but you have to know which.

Two tells: a **uniform image size** across the set (raw Messidor-2 comes in three
resolutions), and a **flat, centred intensity histogram** (raw fundus images are dark
and right-skewed).

In [ ]:
import cv2, numpy as np

m2img = find_mount("messidor2preprocess") or find_mount("messidor2", "preprocess")
Q2_NOTE = "not checked"
if m2img:
    files = [p for p in m2img.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif"}]
    print(f"{len(files)} image files")
    sample = files[:40]
    sizes, means, stds = [], [], []
    for f in sample:
        im = cv2.imread(str(f))
        if im is None:
            continue
        sizes.append(im.shape[:2])
        g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        means.append(g.mean()); stds.append(g.std())
    uniq = set(sizes)
    print("distinct sizes in sample:", uniq if len(uniq) <= 5 else f"{len(uniq)} different")
    print(f"intensity  mean {np.mean(means):.1f}  std {np.mean(stds):.1f}")
    likely = len(uniq) == 1 and np.mean(means) > 80
    Q2_NOTE = ("LIKELY PREPROCESSED (uniform size + high mean intensity)" if likely
               else "likely close to raw (mixed sizes / dark)")
    print("\nverdict:", Q2_NOTE)
    print("Record this in docs/05_dataset_card.md under 'Verification log'.")

---
## Q3 · Do the Messidor-2 adjudicated grades join to the images?

Images and grades are two separate Kaggle datasets. Any image that fails to join is an
image you cannot evaluate on.

In [ ]:
import pandas as pd

Q3_NOTE = "not checked"
m2lab = find_mount("messidor2", "grades") or find_mount("messidor2drgrades")
if m2lab and m2img:
    csvs = list(m2lab.rglob("*.csv"))
    print("label files:", [c.name for c in csvs])
    if csvs:
        df = pd.read_csv(csvs[0])
        print(df.head(3).to_string(index=False))
        print("\ncolumns:", list(df.columns))

        id_col = next((c for c in df.columns
                       if "image" in c.lower() or "id" in c.lower()), df.columns[0])
        label_stems = {Path(str(v)).stem for v in df[id_col]}
        image_stems = {p.stem for p in m2img.rglob("*")
                       if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif"}}
        matched = label_stems & image_stems
        print(f"\njoin on '{id_col}': {len(matched)} matched, "
              f"{len(image_stems - label_stems)} images unlabelled, "
              f"{len(label_stems - image_stems)} labels with no image")
        Q3_NOTE = f"{len(matched)} joined on {id_col}"
        grade_col = next((c for c in df.columns
                          if "grade" in c.lower() or "dr" == c.lower()), None)
        if grade_col:
            print("\ngrade distribution:")
            print(df[grade_col].value_counts().sort_index().to_string())

---
## Q4 · Does IDRiD include the optic-disc and fovea coordinate CSVs?

Experiment C1 trains the geometry regressor on these, and **every quadrant rule in M3
depends on it**. Without a coordinate frame, "haemorrhages in three quadrants" is
undefined and the 4-2-1 rule cannot be evaluated at all.

IDRiD is published in **three parts**, and a grading-only mirror has neither piece
this project needs:

| Part | Contents | Needed for |
|---|---|---|
| A — Segmentation | MA/HE/EX/SE pixel masks + optic-disc masks | C2 (extra mask training data) |
| B — Grading | 516 images + severity CSV | nothing critical |
| **C — Localization** | **optic-disc + fovea centre coordinates** | **C1, and therefore all of M3** |

If the cell below finds no coordinate table, add a mirror carrying Parts A and C —
the dataset titled **Indian Diabetic Retinopathy Image Dataset** has all three.

Part A also contributes **optic-disc masks**, which give M2b a stronger geometry
signal than centre coordinates alone.

In [ ]:
import pandas as pd

idrid = find_any(("idrid",), ("indian", "diabetic", "retinopathy"), label="IDRiD")
idrid_masks, idrid_coords, idrid_seg_images = {}, [], []
Q4_OK = Q4A_OK = False

if idrid:
    parts = sorted({d.name for d in idrid.iterdir() if d.is_dir()})
    print("top-level parts:", parts)

    # --- Part A: lesion + optic-disc masks -------------------------------
    scanned = scan_mask_dirs(idrid)
    for channel in LESION4 + ["optic_disc"]:
        entries = scanned.get(channel, [])
        if entries:
            idrid_masks[channel] = sorted(d for d, _ in entries)
            print(f"  {channel:15s} {sum(n for _, n in entries):5d} masks "
                  f"in {len(entries)} dir(s)")
        else:
            print(f"  {channel:15s}     0 masks")
    Q4A_OK = all(c in idrid_masks for c in LESION4)

    # Original images that pair with those masks.
    idrid_seg_images = sorted(
        str(d) for d in idrid.rglob("*")
        if d.is_dir() and "original" in str(d).lower()
        and "segmentation" in str(d).lower()
        and any(f.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif"} for f in d.iterdir() if f.is_file())
    )
    print("\nPart A original-image dirs:", len(idrid_seg_images))
    for d in idrid_seg_images:
        print("   ", d)

    # --- Part C: optic-disc and fovea centre coordinates -----------------
    tables = [f for f in idrid.rglob("*") if f.suffix.lower() in {".csv", ".xlsx", ".xls"}]
    idrid_coords = [f for f in tables if any(
        k in f.name.lower() for k in ("fovea", "disc", "od_", "centre", "center", "markup"))]
    print(f"\ncoordinate tables: {len(idrid_coords)}")
    for f in idrid_coords[:6]:
        print("   ", f.relative_to(idrid))
    if idrid_coords:
        try:
            f = idrid_coords[0]
            df = pd.read_csv(f) if f.suffix.lower() == ".csv" else pd.read_excel(f)
            print(f"\n{f.name} columns: {list(df.columns)}")
            print(df.head(3).to_string(index=False))
            Q4_OK = True
        except Exception as exc:
            print("  could not read:", exc)

print("\n" + "=" * 62)
print("Q4 (Part C, coordinates) :", "PASS" if Q4_OK else "FAIL")
print("Q4 (Part A, lesion masks):", "PASS" if Q4A_OK else "FAIL")
if Q4_OK and Q4A_OK:
    print("Full IDRiD present. C1 trainable; C2 gets IDRiD masks on top of DDR.")
else:
    if not Q4_OK:
        print("  No optic-disc/fovea coordinates -> C1 cannot be trained, M3 loses")
        print("  its coordinate frame, and the 4-2-1 quadrant rule is unevaluable.")
    if not Q4A_OK:
        print("  No lesion masks -> C2 trains on DDR alone (~757 images).")
    print("  Fix: mount a mirror with Parts A and C, e.g. the dataset titled")
    print("       'Indian Diabetic Retinopathy Image Dataset'.")
print("=" * 62)

---
## Q5 · EyePACS patient IDs and grade distribution

Filenames must parse as `<patient>_<left|right>`. If they don't, patient-grouped
splitting is impossible and every number this project reports would be inflated by
eye-pair leakage (see `docs/02_research_protocol.md` Rule 3).

Two eyes per patient means **images ≈ 2 × patients**. A ratio near 1.0 means the
filenames have been flattened by the mirror and you need a different one.

In [ ]:
from collections import Counter, defaultdict

eyepacs = find_mount("eyepacs")
Q5_OK = False
if eyepacs:
    imgs = [p for p in eyepacs.rglob("*")
            if p.suffix.lower() in {".jpeg", ".jpg", ".png"}]
    print(f"{len(imgs)} images")
    print("layout sample:", *[str(p.relative_to(eyepacs)) for p in imgs[:3]], sep="\n   ")

    # --- patient IDs -----------------------------------------------------
    pat = re.compile(r"^(\d+)_(left|right)$", re.I)
    parsed = [pat.match(p.stem) for p in imgs]
    ok = [m for m in parsed if m]
    patients = {m.group(1) for m in ok}
    print(f"\nparseable stems : {len(ok)}/{len(imgs)}  ({100*len(ok)/max(1,len(imgs)):.1f}%)")
    print(f"unique patients : {len(patients)}")
    print(f"images/patient  : {len(ok)/max(1,len(patients)):.2f}  (expect ~2.0)")
    if not ok:
        print("  example stems:", [p.stem for p in imgs[:5]])

    # --- grade and split can sit at any depth in this mirror -------------
    GRADES, SPLITS = {"0", "1", "2", "3", "4"}, {"train", "val", "valid", "test"}

    def component(path, allowed):
        for part in path.relative_to(eyepacs).parts[:-1]:
            if part.lower() in allowed:
                return part.lower()
        return None

    grades = Counter(component(p, GRADES) for p in imgs)
    splits = Counter(component(p, SPLITS) for p in imgs)

    if any(k for k in grades if k is not None):
        print("\ngrade distribution (from class folders):")
        total = sum(v for k, v in grades.items() if k is not None)
        for g in sorted(k for k in grades if k is not None):
            print(f"  grade {g}: {grades[g]:6d}  ({100*grades[g]/total:5.1f}%)")
        print("\nreference, EyePACS train: 73.5 / 7.0 / 15.1 / 2.5 / 2.0 %")
    else:
        print("\nNo 0-4 class folders found - grades must come from a label file.")
        for f in list(eyepacs.rglob("*.csv"))[:5] + list(eyepacs.rglob("*.txt"))[:5]:
            print("   candidate label file:", f.relative_to(eyepacs))

    # --- is the mirror's own split patient-disjoint? ---------------------
    if any(k for k in splits if k is not None):
        print("\nthis mirror ships its own split:",
              {k: v for k, v in splits.items() if k is not None})
        where = defaultdict(set)
        for p, m in zip(imgs, parsed):
            s = component(p, SPLITS)
            if m and s:
                where[m.group(1)].add(s)
        leaked = [pid for pid, s in where.items() if len(s) > 1]
        print(f"patients appearing in >1 split: {len(leaked)}")
        if leaked:
            print("  *** THIS MIRROR'S SPLIT LEAKS ACROSS PATIENTS ***")
            print("  e.g.", [(pid, sorted(where[pid])) for pid in leaked[:3]])
            print("  Do not adopt it. Build your own patient-grouped split in Phase 2")
            print("  (docs/02_research_protocol.md Rule 3).")
        else:
            print("  no cross-split patients detected in this mirror's own split")

    Q5_OK = len(ok) > 0.95 * len(imgs) and len(ok) / max(1, len(patients)) > 1.5

print("\n" + "=" * 62)
print("Q5", "PASS" if Q5_OK else "FAIL - patient grouping not possible from these filenames")
if not Q5_OK:
    print("  Without <patient>_<left|right> stems, a patient's two eyes cannot be")
    print("  kept together, and every reported number would be inflated by")
    print("  eye-pair leakage. Find a mirror that preserves the original names.")
print("=" * 62)

---
## Summary — paste this into `docs/05_dataset_card.md`

In [ ]:
verdicts = {
    "Q1_ddr_lesion_masks": "PASS" if Q1_OK else "FAIL",
    "Q2_messidor2_preprocessing": Q2_NOTE,
    "Q3_messidor2_grade_join": Q3_NOTE,
    "Q4_idrid_coordinates": "PASS" if Q4_OK else "FAIL",
    "Q5_eyepacs_patient_ids": "PASS" if Q5_OK else "FAIL",
    "ddr_mask_dirs": ddr_masks,
    "ddr_seg_image_dirs": ddr_seg_images,
    "idrid_mask_dirs": idrid_masks,
    "idrid_seg_image_dirs": idrid_seg_images,
    "idrid_coord_tables": [str(f) for f in idrid_coords],
}
Path("/kaggle/working/verification_log.json").write_text(json.dumps(verdicts, indent=2))
print(json.dumps(verdicts, indent=2))

verdicts["Q4a_idrid_lesion_masks"] = "PASS" if Q4A_OK else "FAIL"
blocking = [k for k in ("Q1_ddr_lesion_masks", "Q4_idrid_coordinates",
                        "Q5_eyepacs_patient_ids") if verdicts[k] == "FAIL"]
print("\n" + "=" * 62)
if blocking:
    print("BLOCKED:", ", ".join(blocking))
    print("Resolve these before Phase 1. Do not start the cache build.")
else:
    print("Phase 0 clear. Proceed to 01_build_cache.ipynb.")
print("=" * 62)
print("\nCopy the ddr_mask_dirs paths above into the Phase 1 notebook.")